# GCAM–MARIO · Steel production routes

How much EU steel comes from scrap-based electric arc furnaces (EAF) under GCAM’s `NDC_LTT_CBAM` scenario in 2050? Apply the study’s regional production mix to the full **EXIOBASE Hybrid Green Steel 1.3.0** supply and use table (SUT), then measure the shares in MARIO’s solved supply flows. The scenario changes which activities supply steel in EU-12 and EU-15. Final demand and route-specific production recipes are inputs to this experiment.

The regional GCAM volumes come from the study maintainer’s `SteelProdTech` export. The published supplement reports the corresponding EU27 aggregates. Cite [Ghezzi et al.](https://doi.org/10.1021/acs.est.5c15099) and see the [regional data and provenance](https://github.com/SESAM-Polimi/ExioSteel/tree/be43497a18302f7c2ad0df13a3cb6b6536355839/examples/gcam-output) and [background database](https://zenodo.org/records/16779787).

## Prepare the two inputs

1. Get the [pinned ExioSteel checkout](https://github.com/SESAM-Polimi/ExioSteel/tree/be43497a18302f7c2ad0df13a3cb6b6536355839). The checkout includes the IAMC CSV and the route concordance.
   ```bash
   git clone --branch feature/gcam-shock-workflow https://github.com/SESAM-Polimi/ExioSteel.git
   cd ExioSteel
   git checkout be43497a18302f7c2ad0df13a3cb6b6536355839
   ```
2. If `models/prepared/` already exists from this checkout, reuse it. To create it, use a separate **Python 3.12** environment inside ExioSteel. Run `python tools/install.py` followed by `gcam-mario prepare`. First preparation downloads about 1.78 GB. See the [preparation instructions](https://github.com/SESAM-Polimi/ExioSteel/blob/be43497a18302f7c2ad0df13a3cb6b6536355839/docs/installation.md).
3. Run this notebook in a Jupyter kernel with the [tested MARIO revision on `link-iam-mrio`](https://github.com/it-is-me-mario/MARIO/tree/f4594ab917f75f687b44171b8ce9027ed8c97359). From that MARIO checkout, install it in the kernel environment with `python -m pip install -e .`. The preparation environment uses MARIO 1.0.0. Set `EXIOSTEEL_ROOT` to the ExioSteel checkout before launching Jupyter, or enter its path in the next cell.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mario

root = Path(os.environ.get("EXIOSTEEL_ROOT", ".")).expanduser()
export = root / "examples/gcam-output/steel.csv"
concordance = root / "config/variables.json"
coefficients = root / "models/prepared/coefficients"
if not all(path.is_file() for path in (export, concordance, coefficients / "z.parquet")):
    raise FileNotFoundError("Set root to the prepared ExioSteel checkout (see setup above).")

MODEL, SCENARIO, YEAR = "GCAM 7.0", "NDC_LTT_CBAM", 2050
REGIONS = {"EU-12": "EU-12", "EU-15": "EU-15"}  # MARIO region -> IAM region
UNIT = "Mt/yr"
STEEL = "Basic iron and steel and of ferro-alloys and first products thereof"
SCRAP = "Re-processing of secondary steel into new steel"

mario.list_iam_shocks(implemented_only=True).loc[["steel_mix"], ["family", "operator"]]

## Read the GCAM run and match its routes

`mario.read_iamc` returns an `IamData` with the original model, scenario and variable labels. The study maps 21 series to 18 activities. Four scrap-EAF series distinguish finishing fuel or electricity and supply the same EXIOBASE activity. Direct-reduced iron processed in an EAF is a separate route.

In [ ]:
iam = mario.read_iamc(export)
route_for_variable = json.loads(concordance.read_text(encoding="utf-8"))["steel"]
variables = {}
for variable, activity in route_for_variable.items():
    variables.setdefault(activity, []).append(variable)

print("Available scenarios:", ", ".join(sorted(iam.frame["Scenario"].unique())))
print(f"{len(route_for_variable)} GCAM series → {len(variables)} supply activities")
pd.Series(variables[SCRAP], name="GCAM series assigned to scrap EAF").to_frame()

Calculate an independent target for each activity from tonnes produced. Pool the underlying GCAM volumes before dividing by each region’s total.

In [ ]:
observations = iam.frame.loc[
    (iam.frame["Model"] == MODEL) & (iam.frame["Scenario"] == SCENARIO)
    & (iam.frame["Unit"] == UNIT) & iam.frame["Region"].isin(REGIONS.values())
]
volumes = observations.pivot(index="Variable", columns="Region", values=YEAR)
assert set(volumes.index) == set(route_for_variable), "Check the steel route concordance."
volumes = volumes.loc[:, list(REGIONS.values())]
assert np.isfinite(volumes.to_numpy()).all() and (volumes >= 0).all().all()
production = volumes.groupby(pd.Series(route_for_variable, name="Activity")).sum()
assert (production.sum() > 0).all()
target = production.div(production.sum())
production.sum().rename(f"GCAM steel production ({UNIT})").to_frame()

## Open the hybrid steel market

The prepared SUT contains the route inventories and a common steel commodity measured in tonnes. Each column of `s` gives the activities supplying a commodity. Read the domestic steel column in each modelled region and retain the mapped activities. Normalising these coefficients gives the baseline shares within that group.

In [ ]:
db = mario.parse_from_parquet(path=str(coefficients), table="SUT", mode="coefficients", calc_all=False)
assert db.units["Commodity"].at[STEEL, "unit"] == "tonnes"

def domestic_steel(matrix):
    columns = {}
    for region in REGIONS:
        rows = (region, "Activity", list(variables))
        column = (region, "Commodity", STEEL)
        columns[region] = matrix.loc[rows, column].droplevel([0, 1])
    return pd.DataFrame(columns).reindex(list(variables))

base_coeff = domestic_steel(db.get_block_as_pandas("s", scenario="baseline"))
assert (base_coeff.sum() > 0).all()
baseline = base_coeff.div(base_coeff.sum())

## Apply the steel mix

`shock_from_iam` creates a scenario from the baseline and returns its name. MARIO reads the physical basis from the commodity unit. GCAM’s Mt/yr cancel when calculating shares. MARIO preserves the group’s combined supply coefficient in each selected market and solves the scenario at the MRIO’s own scale.

In [ ]:
case = db.shock_from_iam(
    iam, shock="steel_mix",
    model=MODEL, scenario=SCENARIO, year=YEAR, regions=REGIONS,
    commodity=STEEL, variables=variables, unit=UNIT,
)
print(case)

## Verify all 18 routes in both regions

Read `S`, the supply table in flow units, from the new scenario. Divide each route’s tonnes by the sum across the 18 routes and compare all 36 shares against the target independently calculated from the CSV. Also check that the combined supply coefficient of the mapped activities is preserved in each region.

In [ ]:
scenario_coeff = domestic_steel(db.get_block_as_pandas("s", scenario=case))
np.testing.assert_allclose(scenario_coeff.sum(), base_coeff.sum(), rtol=1e-10)

supply = domestic_steel(db.query("S", scenarios=case))
assert (supply.sum() > 0).all()
realised = supply.div(supply.sum())
expected = target.loc[realised.index, [REGIONS[r] for r in realised.columns]]
np.testing.assert_allclose(realised, expected, rtol=1e-9, atol=1e-12)

pd.DataFrame({
    "Baseline scrap EAF (%)": 100 * baseline.loc[SCRAP],
    "GCAM scrap EAF (%)": 100 * target.loc[SCRAP],
    "MARIO scrap EAF (%)": 100 * realised.loc[SCRAP],
    "MARIO supply across routes (Mt)": supply.sum() / 1e6,
}).round(4)

In [ ]:
# Show the same routes on both panels, ranked by their largest share in either run.
ranking = pd.concat([baseline, realised], axis=1).max(axis=1)
shown = ranking.nlargest(5).sort_values().index
labels = {
    SCRAP: "Scrap EAF",
    "Manufacture of basic iron and steel and of ferro-alloys and first products thereof": "BF–BOF",
}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharex=True, sharey=True)
for ax, region in zip(axes, REGIONS):
    comparison = pd.DataFrame({
        "Baseline": 100 * baseline.loc[shown, region],
        str(YEAR): 100 * realised.loc[shown, region],
    }).rename(index=labels)
    comparison.plot.barh(ax=ax, color=["#a8b3bc", "#08736e"], width=0.74, legend=False)
    ax.set(title=region, xlabel="Share of mapped steel supply (%)", ylabel="")
axes[0].legend(frameon=False)
fig.suptitle(f"Steel production by route · {SCENARIO}")
fig.tight_layout()
plt.show()

### Compare the regional result with EU27

For `NDC_LTT_CBAM` in 2050, the solved scrap-EAF shares round to **71.16% in EU-12** and **71.17% in EU-15**. To compare with the published EU27 grouping, sum the two regions’ GCAM volumes first. The corresponding MARIO aggregate uses its own regional supply volumes as weights.

In [ ]:
pd.Series({
    "EU27 · GCAM": 100 * production.loc[SCRAP].sum() / production.to_numpy().sum(),
    "EU27 · MARIO": 100 * supply.loc[SCRAP].sum() / supply.to_numpy().sum(),
}, name="Scrap EAF (% of mapped steel output)").round(4).to_frame()